Hypotheses: ...

Core Findings: Initial experiment phase showed that when we extract the reasoning tokens from a reasoning llm (qwen3.5:4b - param: reasoning = True) that we hand a sample out of the data set to solve and we inject these reasoning tokens into a non reasoning (qwen3.5:4b params: reasoning = False) the precision is reducing.
Research that comes to similiar results: https://arxiv.org/pdf/2509.23196

Next steps: 
1. Redo experiment with reasoning llm (qwen3.5:4b - param: reasoning = None) this will generate reasoning within tags <think></think>, extract these and inject them into the the next prompts.



## 00 Trying to extract content between think tags

In [2]:
import time
import json
import os
import subprocess
import random
from datetime import datetime
from langchain_ollama import ChatOllama
from datasets import load_dataset
import re
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
# Set llm model parameters using the OpenAI-compatible endpoint
llm_model = ChatOpenAI(
    model="qwen3.5:4b", 
    base_url=f"http://{windows_ip}:11434/v1", 
    api_key="ollama", # LangChain requires an API key string, even if dummy
    temperature=1.0, top_p=0.95, presence_penalty=1.5
    # Note: top_k=20 was removed as it is not natively supported by the standard OpenAI spec.
    # If needed, it can be passed in model_kwargs.
)

In [23]:
# 1. Ask the anchor question
anchor_prompt = "John cannot touch the cup as it was just brought out of the refrigerator. He wanted some air to blow around it. The _ is very warm.\nOptions: cup or air."

# We relax the strict "only" constraint for the anchor to allow the model to think,
# and explicitly prompt for the tags just in case it needs a nudge.
anchor_response = llm_model.invoke([
    SystemMessage(content="Think about the problem and then solve it."),
    HumanMessage(content=anchor_prompt)
])

# 2. Check for the reasoning trace in two places

full_output = anchor_response.content
# Regex to catch thinking tags (case-insensitive, handles newlines)
think_match = re.search(r'<think>(.*?)</think>', full_output, re.DOTALL | re.IGNORECASE)

# Check if the API separated the reasoning into metadata (common for Qwen Max / DeepSeek APIs)
api_reasoning = anchor_response.additional_kwargs.get("reasoning_content", "")

if api_reasoning:
    precomputed_reasoning = api_reasoning.strip()
    print("Successfully extracted reasoning from API metadata (additional_kwargs).")
elif think_match:
    precomputed_reasoning = think_match.group(1).strip()
    print("Successfully extracted reasoning tokens from text tags.")
else:
    precomputed_reasoning = "No reasoning extracted."
    print("Warning: No reasoning trace found.")
    print(f"Raw text output: {full_output}")
    print(f"Raw kwargs: {anchor_response.additional_kwargs}")

Raw text output: The sentence states that John cannot touch the cup because it was just brought out of the refrigerator, which implies the cup is cold.

If the **cup** were very warm, it would contradict the information that it was just taken out of a refrigerator (where it would be cold).

If the **air** is very warm, this makes sense in the context of the cold cup (the refrigerator is cold, the room is warm). John wants air to blow around the cup, likely to cool it further or simply because the warm air is contrasting with the cold cup. The statement "The air is very warm" fits the logical relationship established by the cold cup coming from the fridge.

Therefore, the most logical completion is **air**.

Answer: **air**
Raw kwargs: {'refusal': None}


In [24]:
# 1. Ask the anchor question
anchor_prompt = "John cannot touch the cup as it was just brought out of the refrigerator. He wanted some air to blow around it. The _ is very warm.\nOptions: cup or air."

# We relax the strict "only" constraint for the anchor to allow the model to think,
# and explicitly prompt for the tags just in case it needs a nudge.
anchor_response = llm_model.invoke([
    SystemMessage(content="Solve the problem step-by-step. You MUST output your internal reasoning entirely in English inside <think> </think> tags before providing the final answer."),
    HumanMessage(content=anchor_prompt)
])

# 2. Check for the reasoning trace in two places

full_output = anchor_response.content
# Regex to catch thinking tags (case-insensitive, handles newlines)
think_match = re.search(r'<think>(.*?)</think>', full_output, re.DOTALL | re.IGNORECASE)

# Check if the API separated the reasoning into metadata (common for Qwen Max / DeepSeek APIs)
api_reasoning = anchor_response.additional_kwargs.get("reasoning_content", "")

if api_reasoning:
    precomputed_reasoning = api_reasoning.strip()
    print("Successfully extracted reasoning from API metadata (additional_kwargs).")
elif think_match:
    precomputed_reasoning = think_match.group(1).strip()
    print("Successfully extracted reasoning tokens from text tags.")
else:
    precomputed_reasoning = "No reasoning extracted."
    print("Warning: No reasoning trace found.")
    print(f"Raw text output: {full_output}")
    print(f"Raw kwargs: {anchor_response.additional_kwargs}")

Raw text output: Let's analyze the prompt step by step to determine the correct answer.

### 1. Breakdown of the Premises

The provided text is:
> "S, Cup is not Warm. Therefore, something else is Warm."

The prompt asks if "cup" is the answer, based on a linguistic trick.

### 2. Analysis of the Linguistic Trick

The prompt presents a scenario where a "linguistic trick" might make "cup" seem plausible or incorrect.

*   **Premise 1:** "S, Cup is not Warm."
    *   This statement explicitly states (or implies via common sense) that the cup is **not warm**.
    *   If we assume the premise "S" stands for the subject of the second sentence, and the second sentence is "The _ is very warm", then the subject ("_") cannot be "cup" because "cup" is already established as not warm.
*   **Premise 2:** "Therefore, something else is Warm."
    *   This is a logical deduction. If X (the cup) is not warm, and Y (the blank) is warm, then Y must be something other than X.
    *   This logic rules out